# Translating an MF6 output folder into FloPy code

## Introduction

The purpose of this exercise is to take an **existing MODFLOW 6 model** (a
folder of `mfsim.nam` + package input files) and turn it back into a clean,
readable **FloPy build script** made of explicit constructor calls, e.g.

```python
ic = flopy.mf6.ModflowGwfic(gwf, pname="ic", strt=strt)
```

We will:

1. check that the required Python packages are available,
2. point the tool at an existing MF6 model folder,
3. generate a FloPy build script from it,
4. inspect the generated code,
5. run the generated script to **rebuild** the model (a full round-trip),
6. optionally add plots and GIS exports, and
7. generate a full **build → run → post-process** workflow.

The heavy lifting is done by `mf6_to_flopy.py`, which sits one directory up from
this notebook. It is package-agnostic and also auto-repairs foreign/absolute
paths that ModelMuse or GMS sometimes bake into the name files.

## 1. Dependency check

Core packages are `numpy` and `flopy`; `matplotlib`, `pandas`, `geopandas` and
`vtk` are only needed for the optional plot / post-process / export steps.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join('..')))
import mf6_to_flopy as m2f

m2f.check_dependencies(need_plots=True, need_export=True, need_postprocess=True)
print('flopy is available -- ready to go.')

flopy is available -- ready to go.


## 2. Point at an existing MF6 model folder

Set `model_folder` to a directory that contains `mfsim.nam` and the package
files. Here we use the bundled `examples/01_structured_dis`, but you can replace
this with the path to your own MF6 model (for example a ModelMuse export).

In [2]:
model_folder = os.path.join('..', 'examples', '01_structured_dis')
assert os.path.isdir(model_folder), f'not a folder: {model_folder}'
print('MF6 files found in', model_folder, ':')
for f in sorted(os.listdir(model_folder)):
    print('   ', f)

MF6 files found in ../examples/01_structured_dis :
    mfsim.nam
    symp01.dis
    symp01.ic
    symp01.ims
    symp01.nam
    symp01.npf
    symp01.oc
    symp01.rcha
    symp01.riv
    symp01.tdis


## 3. Generate the FloPy build script

Every CLI flag maps to a keyword argument on `generate_script()`:

| argument         | CLI flag          | effect                                            |
|------------------|-------------------|---------------------------------------------------|
| `add_plots`      | `--plots`         | append head-contour plotting code                 |
| `add_export`     | `--export`        | append grid + attribute shapefile export code     |
| `add_vtk`        | `--vtk`           | also export the model to VTK (needs `vtk`)         |
| `keep_external`  | `--keep-external` | keep `OPEN/CLOSE` refs instead of inlining data   |
| `add_run`        | `--run`           | append `sim.run_simulation()`                     |
| `add_postprocess`| `--postprocess`   | full workflow: run + heads + specific discharge + List Budget |

In [3]:
out_script = 'build_model.py'

m2f.generate_script(
    model_folder,
    out_path=out_script,
    sim_ws='rebuilt',
    add_plots=True,
    add_export=True,
    add_vtk=False,
    keep_external=False,
    add_run=False,        # set True (or use add_postprocess) to run the model
    add_postprocess=False,
)

Loading MF6 simulation from: ../examples/01_structured_dis
Wrote FloPy build script -> build_model.py
  simulation     : modflowsim
  models         : symp01
  stress periods : 1
  options        : plots=True, export=True, vtk=False, keep_external=False, postprocess=False, run=False


'build_model.py'

## 4. Inspect the generated code

The output is ordinary, readable FloPy: one explicit constructor per package,
an automatic `sim.register_ims_package(...)` per solver, and (if present)
exchange packages with `exgtype` / `exgmnamea` / `exgmnameb`.

In [4]:
print(open(out_script, encoding='utf-8').read())

"""
Auto-generated FloPy build script.
Source model: /home/claude/mf6_to_flopy_project/examples/01_structured_dis
Generated by mf6_to_flopy.py
"""
import os
import flopy
import numpy as np
import matplotlib.pyplot as plt

sim_ws = 'rebuilt'
os.makedirs(sim_ws, exist_ok=True)

# ------------------------------------------------------------------
# Simulation
# ------------------------------------------------------------------
sim = flopy.mf6.MFSimulation(
    sim_name='modflowsim',
    version="mf6",
    exe_name='mf6',
    sim_ws=sim_ws,
)

# --- simulation package: ModflowTdis ---
tdis = flopy.mf6.ModflowTdis(
    sim,
    time_units='days',
    nper=1,
    perioddata=[[1, 1, 1]],
)

# --- simulation package: ModflowIms ---
ims = flopy.mf6.ModflowIms(
    sim,
    pname='symp01',
    complexity='simple',
    linear_acceleration='bicgstab',
)

# --- register solvers with their model(s) ---
sim.register_ims_package(ims, ['symp01'])

# -----------------------------------------------------

## 5. Round-trip: run the generated script to rebuild the model

Executing the script rebuilds the simulation and calls `sim.write_simulation()`,
writing a fresh set of MF6 input files into the `rebuilt` workspace.

In [5]:
import runpy
runpy.run_path(out_script, run_name='__main__')

print('\nRebuilt files:')
for f in sorted(os.listdir('rebuilt')):
    print('   ', f)

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package symp01...
  writing model symp01...
    writing model name file...
    writing package dis...
    writing package ic...
    writing package npf...
    writing package rch...
    writing package riv_0...
INFORMATION: maxbound in ('', 'riv', 'dimensions') changed to 10 based on size of stress_period_data
    writing package oc...
Head plotting skipped: 'NoneType' object has no attribute 'get_data'


Wrote symp01_grid.shp
Wrote symp01_attrs.shp

Rebuilt files:
    gis
    mfsim.nam
    modflowsim.ims
    modflowsim.tdis
    symp01.dis
    symp01.ic
    symp01.nam
    symp01.npf
    symp01.oc
    symp01.rcha
    symp01.riv


/opt/pyvenv/lib/python3.12/site-packages/pyogrio/geopandas.py:948: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
/opt/oai-pkgs/flopy/export/utils.py:620: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(f)
/opt/pyvenv/lib/python3.12/site-packages/pyogrio/geopandas.py:948: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


## 6. (Optional) Verify the round-trip

Load both the original and the rebuilt model and compare an array; they should
match to floating-point precision.

In [6]:
import numpy as np
import flopy

orig = flopy.mf6.MFSimulation.load(sim_ws=model_folder, verbosity_level=0)
rebu = flopy.mf6.MFSimulation.load(sim_ws='rebuilt', verbosity_level=0)
mname = orig.model_names[0]
top_o = orig.get_model(mname).dis.top.get_data()
top_r = rebu.get_model(mname).dis.top.get_data()
print('DIS top max abs diff :', float(np.max(np.abs(top_o - top_r))))

DIS top max abs diff : 0.0


## 7. Full workflow: build → run → post-process

Regenerate the script with `add_postprocess=True`. This appends, after the
build:

1. **Run** — `sim.run_simulation()` with a success check.
2. **Heads** — reads `gwf.output.head()` and writes a contoured map per layer
   (`head_layer_*.png`).
3. **Specific discharge** — computes vectors with
   `flopy.utils.postprocessing.get_specific_discharge(...)` and plots them
   (`specific_discharge.png`).
4. **List Budget** — reads the `.lst` file with `flopy.utils.Mf6ListBudget`,
   prints the incremental & cumulative budgets, and saves them as CSVs.

If the source model has no Output Control (OC) package, the tool auto-adds one
so heads and budget are recorded.

> **Note:** running requires the `mf6` executable on your PATH. If it is not
> available, the run/plot/budget steps print a friendly "skipped" message and
> the notebook continues.

In [7]:
m2f.generate_script(
    model_folder,
    out_path='build_model_full.py',
    sim_ws='rebuilt_full',
    add_postprocess=True,   # run + heads + specific discharge + List Budget
)
print(open('build_model_full.py', encoding='utf-8').read())

Loading MF6 simulation from: ../examples/01_structured_dis
Wrote FloPy build script -> build_model_full.py
  simulation     : modflowsim
  models         : symp01
  stress periods : 1
  options        : plots=False, export=False, vtk=False, keep_external=False, postprocess=True, run=True
"""
Auto-generated FloPy build script.
Source model: /home/claude/mf6_to_flopy_project/examples/01_structured_dis
Generated by mf6_to_flopy.py
"""
import os
import flopy
import numpy as np
import matplotlib.pyplot as plt

sim_ws = 'rebuilt_full'
os.makedirs(sim_ws, exist_ok=True)

# ------------------------------------------------------------------
# Simulation
# ------------------------------------------------------------------
sim = flopy.mf6.MFSimulation(
    sim_name='modflowsim',
    version="mf6",
    exe_name='mf6',
    sim_ws=sim_ws,
)

# --- simulation package: ModflowTdis ---
tdis = flopy.mf6.ModflowTdis(
    sim,
    time_units='days',
    nper=1,
    perioddata=[[1, 1, 1]],
)

# --- simulat

In [8]:
# Execute the full workflow. Requires mf6 on PATH to actually run/plot/budget;
# otherwise the run steps are skipped gracefully.
import runpy
try:
    runpy.run_path('build_model_full.py', run_name='__main__')
except Exception as exc:
    print('Full workflow could not complete (mf6 on PATH?):', exc)

# Show any post-processing artifacts that were produced
import glob
for pat in ('rebuilt_full/head_layer_*.png', 'rebuilt_full/specific_discharge.png',
            'rebuilt_full/budget_*.csv'):
    for f in sorted(glob.glob(pat)):
        print('  produced:', f)

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package symp01...
  writing model symp01...
    writing model name file...
    writing package dis...
    writing package ic...
    writing package npf...
    writing package rch...
    writing package riv_0...
INFORMATION: maxbound in ('', 'riv', 'dimensions') changed to 10 based on size of stress_period_data
    writing package oc...
Full workflow could not complete (mf6 on PATH?): The program mf6 does not exist or is not executable.


## 8. Try the other examples

The project ships four example models. Swap `model_folder` above for any of
these and re-run:

* `examples/01_structured_dis`     -- structured DIS grid (this notebook default)
* `examples/02_unstructured_disv`  -- unstructured DISV (vertex) grid
* `examples/03_unstructured_disu`  -- fully unstructured DISU grid
* `examples/04_coupled_gwf_gwt`    -- coupled GWF-GWT (flow + solute transport)

## Summary

We loaded an existing MF6 model, translated it into an explicit FloPy build
script, inspected the code, rebuilt the model, and generated a full
build → run → post-process workflow. The same steps are available from the
command line:

```bat
python mf6_to_flopy.py "<MODEL_FOLDER>" -o build_model.py --postprocess
python build_model.py
```